In [1]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    "../data/processed/insurance_data_cleaned.csv"
)

C:\Users\mijuu\AppData\Local\Temp\ipykernel_24392\2484231375.py:1: DtypeWarning: Columns (0: mmcode, 1: Cylinders, 2: cubiccapacity, 3: kilowatts, 4: NumberOfDoors, 5: CapitalOutstanding) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Claim Indicator

Needed for frequency analysis.

In [3]:
df['HasClaim'] = np.where(
    df['TotalClaims'] > 0,
    1,
    0
)

Margin

In [4]:
df['Margin'] = (
    df['TotalPremium'] - df['TotalClaims']
)

KPIS

Claim Severity

In [5]:
df['Province'].value_counts()

Province
Gauteng          393865
Western Cape     170796
KwaZulu-Natal    169781
North West       143287
Mpumalanga        52718
Eastern Cape      30336
Limpopo           24836
Free State         8099
Northern Cape      6380
Name: count, dtype: int64

In [6]:
gauteng = df[
    df['Province'] == 'Gauteng'
]['TotalClaims']

western_cape = df[
    df['Province'] == 'Western Cape'
]['TotalClaims']

Run T-Test

In [8]:
t_stat, p_value = ttest_ind(
    gauteng,
    western_cape,
    equal_var=False
)

print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

P-value: 0.06215231452280036
Fail to reject H0


Hypothesis 2
H0: No Risk Difference Between Zip Codes

In [9]:
df['PostalCode'].value_counts().head()

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
Name: count, dtype: int64

In [10]:
zip1 = df[
    df['PostalCode'] == 2000
]['TotalClaims']

zip2 = df[
    df['PostalCode'] == 122
]['TotalClaims']

T-Test

In [11]:
t_stat, p_value = ttest_ind(
    zip1,
    zip2,
    equal_var=False
)

print(p_value)

0.5457671850912713


Hypothesis 4
H₀: No Risk Difference Between Women & Men

In [12]:
#Contingency Table

gender_claim = pd.crosstab(
    df['Gender'],
    df['HasClaim']
)

gender_claim

HasClaim,0,1
Gender,,
Female,6741,14
Male,42723,94
Not specified,938324,2666
Unknown,9522,14


Chi-Square Test

In [13]:
chi2, p_value, dof, expected = chi2_contingency(
    gender_claim
)

print("P-value:", p_value)

P-value: 0.003993781054238153


In [ ]:
results = pd.DataFrame({
    'Hypothesis': [
        'Province Risk Difference',
        'Zip Code Risk Difference',
        'Gender Risk Difference'
    ],
    
    'Test': [
        'T-Test',
        'T-Test',
        'Chi-Square'
    ],
    
    'P-Value': [
        0.06215231452280036,
        0.5457671850912713,
        0.003993781054238153
    ]
})

results['Decision'] = np.where(
    results['P-Value'] < 0.05,
    'Reject H0',
    'Fail to Reject H0'
)

results